# EDA – steg 5 til 8 (mal)

Malen følger stegene 5–8 i GeeksforGeeks, *What is Exploratory Data Analysis?*
(https://www.geeksforgeeks.org/data-analysis/what-is-exploratory-data-analysis/):

| Steg | Tema | Seksjon |
|---|---|---|
| 5 | Performing Data Transformation | [5](#5-datatransformasjon) |
| 6 | Visualizing Relationship of Data | [6](#6-visualisering-av-sammenhenger) |
| 7 | Handling Outliers | [7](#7-håndtering-av-outliers) |
| 8 | Communicate Findings and Insights | [8](#8-kommunisere-funn-og-innsikt) |

**Slik bruker dere malen**
- Hver seksjon starter med *hva steget går ut på* og *spørsmål dere bør svare på*.
- Kodeceller med `# TODO` skal dere fylle ut selv. Hintene peker på funksjoner og dokumentasjon, ikke ferdige løsninger.
- Etter hvert resultat står en **Tolkning**-celle. Skriv 1–3 setninger der. Det er disse som blir til rapportteksten.
- Steg 1–4 (forstå data, inspisere, manglende verdier, egenskaper) ligger allerede delvis i `eda.ipynb`. Steg 5 forutsetter at de er gjort.

> GeeksforGeeks er skrevet for generell EDA, ofte med maskinlæring som mål. Vårt mål er å **designe et databaseskjema og rensing før innsetting i MySQL**. Vurder hvert punkt i lys av det, og skriv kort hvorfor et punkt er relevant eller ikke.

## Oppsett

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

FIG_DIR = Path("figures")       # lagre plott her, så de kan brukes i rapporten
FIG_DIR.mkdir(exist_ok=True)

# Hele filen er ~1,9 GB. Utvikle på et utvalg, kjør på alt til slutt.
NROWS = 100_000                 # sett til None for hele datasettet

In [ ]:
# TODO: les inn porto.csv (bruk NROWS). Vurder dtype= og usecols= for å spare minne.
# Les: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html

---
# 5. Datatransformasjon

Artikkelen nevner: **skalering** (min-max, standardisering), **encoding** av kategorier (one-hot, label),
**matematiske transformasjoner** (f.eks. log), **avledede features** og **aggregering**.

For oss er de to siste viktigst: de fleste spørsmålene i Part 2 handler om størrelser som ikke finnes som kolonner
(varighet, distanse, sluttid, start/slutt-punkt). Det dere finner ut her, avgjør også hva som bør forhåndsberegnes i databasen.

## 5.1 Typekonvertering

- `TIMESTAMP` er Unix-sekunder. Konverter til datetime. **Hvilken tidssone?** Datasettkilden sier hva tidsstemplet betyr. Velg og skriv ned.
- `CALL_TYPE` og `DAY_TYPE` har få unike verdier. Hva sparer dere på `category`-dtype? (`df.memory_usage(deep=True)` før/etter)
- `MISSING_DATA`: er den faktisk bool, eller tekst `"True"`/`"False"`?
- `ORIGIN_CALL` / `ORIGIN_STAND` er float pga. NaN. Hva bør typen være i MySQL?

In [ ]:
# TODO: TIMESTAMP -> datetime (pd.to_datetime, unit="s", eventuelt tz_localize/tz_convert)
# TODO: kategoriske kolonner -> "category", sammenlign minnebruk før/etter

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

## 5.2 Parse POLYLINE

POLYLINE er en JSON-streng. Den må bli en liste av `[lon, lat]`-par før dere kan regne på den.

- Merk rekkefølgen: **lengdegrad først**, så breddegrad.
- Hvor lang tid tar parsingen på utvalget? Hva betyr det for hele datasettet og for innsettingsskriptet?

In [ ]:
# TODO: parse POLYLINE med json.loads (Series.apply). Mål tiden med %%time eller time.perf_counter.

## 5.3 Avledede features

Lag én kolonne om gangen og sjekk resultatet (`describe()`, noen eksempelrader) før dere går videre.

| Feature | Hint | Brukes i |
|---|---|---|
| `n_points` | lengden av lista | Part 2 oppg. 1, 7 |
| `duration_s` | punktene er tatt hvert **15. sekund**. Er varigheten `n·15` eller `(n−1)·15`? Sjekk definisjonen i datasettkilden. | 4b, 5, 8, 10 |
| `start_time`, `end_time` | `TIMESTAMP` + varighet (`pd.to_timedelta`) | 8, 10 |
| `start_lon/lat`, `end_lon/lat` | første og siste element. Hva skjer med tomme lister? | 9 |
| `hour`, `weekday`, `month` | `.dt`-accessoren | 4b, steg 6 |
| `crosses_midnight` | sammenlign *dato* for start og slutt | 8 |
| `distance_km` | sum av haversine mellom påfølgende punkter (`haversine`-pakken står i `requirements.txt`) | 4b, 5 |
| `avg_speed_kmh` | distanse / varighet. Hva med varighet 0? | steg 7 |

Distanse er den dyreste. Skriv en funksjon som tar én polyline og returnerer km, test den på 2–3 turer for hånd, og kjør den deretter på utvalget.

In [ ]:
# TODO: n_points og duration_s

In [ ]:
# TODO: start_time, end_time, hour, weekday, month, crosses_midnight

In [ ]:
# TODO: start/slutt-koordinater (håndter tomme polylines)

In [ ]:
# TODO: funksjon polyline_distance_km(polyline) -> float
#   Les: https://pypi.org/project/haversine/  (merk: haversine() forventer (lat, lon), POLYLINE har [lon, lat])

In [ ]:
# TODO: avg_speed_kmh

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

## 5.4 Aggregering

Aggreger til nivåene Part 2 spør om:
- **Per taxi:** antall turer, total varighet, total distanse.
- **Per tid:** antall turer per time, ukedag, måned.
- **Per `CALL_TYPE`:** snitt varighet og distanse.

Hint: `groupby(...).agg(...)` med navngitte aggregeringer.
Les: https://pandas.pydata.org/docs/user_guide/groupby.html

In [ ]:
# TODO: per taxi

In [ ]:
# TODO: per time / ukedag / måned

In [ ]:
# TODO: per CALL_TYPE

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

## 5.5 Matematiske transformasjoner

Variabler som `n_points`, `distance_km` og turer per taxi er trolig sterkt høyreskjeve.
- Sjekk med `.skew()`.
- Vil en **log-transformasjon** (`np.log1p`) eller log-akse i plottet gjøre fordelingen lettere å lese?
- Merk: dette er for *analyse og plott*, ikke noe som skal lagres i databasen.

In [ ]:
# TODO: skew() før/etter log1p for de numeriske avledede kolonnene

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

## 5.6 Skalering og encoding – relevant for oss?

Artikkelen beskriver min-max/standardisering og one-hot/label-encoding. Svar kort:
- Trenger vi skalering når målet er en database og SQL-spørringer, ikke en ML-modell?
- Hvordan bør `CALL_TYPE` (`A/B/C`) og `DAY_TYPE` lagres i MySQL? (`ENUM`, `CHAR(1)`, egen oppslagstabell?) Er dette en form for encoding?
- Når *kunne* standardisering vært nyttig hos oss? (Tips: z-score i steg 7.)

**Svar:** _…_

---
# 6. Visualisering av sammenhenger

Artikkelen deler inn i: **kategoriske** variabler (stolpe/kake), **fordelinger** (histogram/boksplott) og
**sammenhenger** mellom variabler (scatter). Vi legger til **tid** og **geografi**, fordi det er det datasettet handler om.

**Sjekkliste for hvert plott (rapportkvalitet):**
tittel · aksenavn med enhet · lesbar skala (log ved skjeve data) · `fig.savefig(FIG_DIR / "navn.png", dpi=150, bbox_inches="tight")` · en tolkning under.

## 6.1 Kategoriske variabler
`CALL_TYPE`, `DAY_TYPE`, `MISSING_DATA`. Stolpediagram av `value_counts()`. Vis gjerne prosent i tillegg til antall.

In [ ]:
# TODO: stolpediagrammer for de kategoriske kolonnene (gjerne plt.subplots(1, 3))

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

## 6.2 Fordelinger av numeriske variabler
`n_points`, `duration_s`, `distance_km`, `avg_speed_kmh`, turer per taxi.
- Histogram **og** boksplott for hver. Hva viser boksplottet som histogrammet skjuler?
- Prøv både lineær og log-akse.

In [ ]:
# TODO: histogram + boksplott

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

## 6.3 Tidsmønstre
- Turer per time på døgnet, per ukedag og per måned.
- **Heatmap** time × ukedag (`pd.crosstab` + `plt.imshow`).
- Stemmer tidsspennet med juli 2013–juni 2014? Ser dere spor av ferier/helligdager (jf. `DAY_TYPE`)?

In [ ]:
# TODO: turer per time / ukedag / måned

In [ ]:
# TODO: heatmap time x ukedag

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

## 6.4 Sammenhenger mellom variabler
- `distance_km` mot `duration_s` (scatter, små og gjennomsiktige punkter). Hvilke punkter ligger langt fra hovedtrenden?
- `duration_s` / `distance_km` per `CALL_TYPE` (boksplott gruppert).
- Tidsbånd (00–06, 06–12, 12–18, 18–24) × `CALL_TYPE` (`pd.crosstab(..., normalize="index")`). Dette forbereder Part 2 oppg. 4b.
- Korrelasjonsmatrise for de numeriske kolonnene (`df[cols].corr()`). Prøv også `method="spearman"`: hvorfor kan den gi et annet svar ved skjeve data?

In [ ]:
# TODO: scatter distanse vs varighet

In [ ]:
# TODO: boksplott per CALL_TYPE

In [ ]:
# TODO: tidsbånd x CALL_TYPE

In [ ]:
# TODO: korrelasjonsmatrise (pearson og spearman)

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

## 6.5 Geografi
Bygg videre på `claude_output/viz_eksempel.py` (punktsky med `explode`).
- Ett delplott per `CALL_TYPE` (`plt.subplots(1, 3, sharex=True, sharey=True)`), gjerne like store utvalg.
- Startpunkter vs. sluttpunkter.
- Tetthet med `hexbin` eller `hist2d` og log-fargeskala.
- **Zoom ut** uten akse-begrensning: finnes det punkter langt utenfor Porto? (tas videre i steg 7)

In [ ]:
# TODO: punktsky per CALL_TYPE

In [ ]:
# TODO: start- vs sluttpunkter / tetthetskart

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

---
# 7. Håndtering av outliers

Artikkelen nevner **IQR**, **z-score**, **domenekunnskap**, og tiltak som **capping** eller **transformasjon**.

Skill mellom tre typer før dere bestemmer noe:
1. **Feil i data** (GPS-hopp, koordinater i havet, umulig hastighet) → kandidat for fjerning/retting.
2. **Ugyldige turer etter oppgavens definisjon** (f.eks. < 3 punkter, Part 2 oppg. 7) → dere må fortsatt kunne *telle* dem.
3. **Sjeldne, men gyldige** turer (lang tur til flyplassen) → bør beholdes.

## 7.1 Statistisk: IQR
Skriv en liten funksjon som returnerer nedre/øvre grense (`Q1 − 1.5·IQR`, `Q3 + 1.5·IQR`) og antall/andel utenfor,
og bruk den på `n_points`, `duration_s`, `distance_km`, `avg_speed_kmh`.
Les: https://pandas.pydata.org/docs/reference/api/pandas.Series.quantile.html

In [ ]:
# TODO: iqr_bounds(series) og en oversiktstabell over alle kolonnene

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

## 7.2 Statistisk: z-score
Regn ut |z| > 3 for de samme kolonnene og sammenlign med IQR.
- Z-score antar omtrent normalfordelte data. Holder det her (se skew fra 5.5)?
- Gir z-score på `log1p`-transformerte verdier et mer fornuftig resultat?

In [ ]:
# TODO: z-score, sammenlign antall flagget med IQR

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

## 7.3 Domeneregler
Statistikk alene sier ikke hva som er *feil*. Lag regler basert på hva som er fysisk mulig:
- **Koordinater utenfor et rimelig område** rundt Porto. Velg en bounding box og begrunn den.
- **GPS-hopp:** avstand mellom *påfølgende* punkter / 15 s. Hvilken fartsgrense er urealistisk for en taxi i by/motorvei?
- **Turer med 0, 1 eller 2 punkter.**
- **Turer med distanse ≈ 0** men lang varighet (står stille med taksameter på?).
- **Svært lange turer** (timer). Feil, eller gyldig?
- **`MISSING_DATA = True`**: hvordan ser de ut sammenlignet med resten?
- **Dupliserte `TRIP_ID`**.

In [ ]:
# TODO: bounding box-sjekk på alle punkter

In [ ]:
# TODO: maks hastighet mellom påfølgende punkter per tur

In [ ]:
# TODO: tell turer med 0/1/2 punkter, distanse≈0, ekstrem varighet, MISSING_DATA, duplikater

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

## 7.4 Beslutning per outlier-type

Fyll ut tabellen. Alternativer: **fjern**, **flagg** (behold med kolonne), **cap**, **rett** (f.eks. fjern enkeltpunkter), **behold**.

| Type | Regel / terskel | Antall (andel) | Tiltak | Begrunnelse |
|---|---|---|---|---|
| Koordinater utenfor område | | | | |
| GPS-hopp | | | | |
| < 3 punkter | | | | |
| Distanse ≈ 0 | | | | |
| Ekstrem varighet | | | | |
| `MISSING_DATA = True` | | | | |
| Duplikat `TRIP_ID` | | | | |

Husk: Part 2 oppg. 7 spør om antall turer med < 3 punkter. Velger dere å fjerne dem, hvordan svarer dere på oppgaven?

## 7.5 Effekt av rensingen
Sammenlign `describe()` og et par plott fra steg 6 **før og etter** reglene over. Hvor mange turer og punkter forsvinner?

In [ ]:
# TODO: anvend reglene på en kopi, sammenlign før/etter

**Tolkning:** _Hva viser resultatet over? Hva betyr det for rensing, skjema eller spørringer?_

---
# 8. Kommunisere funn og innsikt

Artikkelen anbefaler å presentere: **mål og kontekst**, **visualiseringer som støtter funnene**, **nøkkelfunn**,
**begrensninger** og **anbefalinger / videre arbeid**.

Denne seksjonen er utkastet til EDA-delen av rapporten (Results, Part 1). Skriv med egne ord, kort og konkret,
og henvis til figurene i `figures/`.

## 8.1 Mål og kontekst
_Hva var målet med EDA-en? Hvilket datasett (kilde, periode, antall rader), og kjørte dere på hele eller et utvalg?_

## 8.2 Nøkkelfunn
_3–7 punkter. Hvert punkt: funn + tall + figur._

- _…_

## 8.3 Renseregler
_Den endelige lista fra 7.4, i den rekkefølgen innsettingsskriptet skal bruke dem._

## 8.4 Konsekvenser for skjema og innsetting
- _Hvilke avledede verdier fra 5.3 bør forhåndsberegnes og lagres?_
- _Hvordan lagres POLYLINE (egen punkttabell, JSON, begge)?_
- _Datatyper og tidssone._

## 8.5 Begrensninger
_Utvalg vs. hele datasettet, antakelser (15 s-intervall, tidssone), terskler som er skjønnsmessige, …_

## 8.6 Anbefalinger og videre arbeid
_Hva ville dere undersøkt mer med mer tid?_